Groq Libraries

In [ ]:
!pip install groq -q
print("Libraries installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.6 MB/s eta 0:00:00
Libraries installed successfully


In [ ]:
import sqlite3
import pandas as pd
import os

from groq import Groq
import re
print("All Libraries are imported successfully")

All Libraries are imported successfully


In [ ]:
import os
os.environ["GROQ_API_KEY"] = "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"
client = Groq(api_key = os.environ["GROQ_API_KEY"])
Model = "llama-3.1-8b-instant"
print("Groq initialized successfully")
print(f"Using Model: {Model}")

Groq initialized successfully
Using Model: llama-3.1-8b-instant


In [ ]:
import io
csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""
df=pd.read_csv(io.StringIO(csv_data))

print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
print("\nFirst 5 rows")
df.head()



Dataset loaded: 30 rows, 8 columns

First 5 rows


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [ ]:
conn = sqlite3.connect("college.db")
df.to_sql("students", conn, if_exists = "replace", index = False)
print("Database : college.db")
print("table 'students' created with 30 Studenst records")
test_df = pd.read_sql_query("SELECT COUNT(*) as total_rows From students", conn)
print(f"\nVerification: {test_df['total_rows'][0]}rows in database")

Database : college.db
table 'students' created with 30 Studenst records

Verification: 30rows in database


In [ ]:
def get_schema(conn, table_name = "students"):
  cursor = conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns = cursor.fetchall()

  schema_lines = [f"Table: {table_name}"]
  schema_lines.append("Columns:")

  for col in columns:
    schema_lines.append(f" -{col[1]} ({col[2]})")
  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_row = cursor.fetchall()
  schema_lines.append("\nSample Rows (first 3):")

  for row in sample_row:
    schema_lines.append(f" {row}")
  return "\n".join(schema_lines)
schema = get_schema(conn)
print(schema)

Table: students
Columns:
 -student_id (INTEGER)
 -name (TEXT)
 -age (INTEGER)
 -gender (TEXT)
 -subject (TEXT)
 -marks (INTEGER)
 -attendance (INTEGER)
 -grade (TEXT)

Sample Rows (first 3):
 (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
 (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
 (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [ ]:
def generate_sql(user_question, schema_text, client, model):
  system_prompt = f"""You are an expert SQL assistant.
  You are connected to a SQLite database with the following structure:


  {schema_text}


  Rules you must follow:
  1. Generate ONLY a valid SQLite SQL query.
  2. Do not include any explanation or text — only the SQL query.
  3. Do not use markdown code blocks. Return the raw SQL only.
  4. The table name is: students
  5. Only use column names that exist in the schema above.
  6. Use single quotes for string values in WHERE clauses (example: WHERE subject = 'Programming').
  7. If the user asks for top N, use ORDER BY marks DESC LIMIT N.
  """

  response = client.chat.completions.create(
      model = "llama-3.1-8b-instant",
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_question}
      ],
      temperature = 0.0
  )
  sql_query = response.choices[0].message.content.strip()
  return sql_query
question = "Show me all female students"
print(f"Question: {question}")
sql = generate_sql(question, schema, client, Model)
print(f"SQL Query: {sql}")

Question: Show me all female students
SQL Query: SELECT * FROM students WHERE gender = 'Female'


In [ ]:
def execute_sql(sql_query, conn):
  clean_sql = sql_query.strip()
  clean_sql = re.sub(r'```sql\s*', '', clean_sql)
  clean_sql = re.sub(r'```s*', '', clean_sql)
  clean_sql = clean_sql.strip()
  try:
    result_df = pd.read_sql_query(clean_sql, conn)
    return result_df, None
  except Exception as e:
    return None, str(e)
print(f"Executing SQL: {sql}")
result, error = execute_sql(sql, conn)
if error:
  print(f"Error executing SQL: {error}")
else:
  print(f"\nQuery Returned {len(result)} rows:")
  print(result)

Executing SQL: SELECT * FROM students WHERE gender = 'Female'

Query Returned 15 rows:
    student_id            name  age  gender      subject  marks  attendance  \
0            2     Priya Patel   21  Female      Science     76          85   
1            4      Sneha Iyer   22  Female  Mathematics     62          78   
2            6  Divya Krishnan   20  Female      Science     83          88   
3            8    Ananya Gupta   21  Female  Programming     89          96   
4           10    Pooja Sharma   22  Female  Mathematics     55          72   
5           12   Meera Nambiar   20  Female      Science     81          87   
6           14   Kavitha Rajan   21  Female  Programming     86          93   
7           16   Swathi Pillai   22  Female  Mathematics     90          95   
8           18   Lavanya Menon   20  Female      Science     66          76   
9           20    Anjali Singh   21  Female  Programming     94          97   
10          22    Rekha Sharma   22  Female 

### Fixing `NameError` by re-defining essential variables

The `NameError` often occurs if the kernel has restarted, causing previously defined variables (`client`, `model`, `df`, `conn`) to be lost. The following cells explicitly re-define these variables to ensure the `text_to_sql_agent` function can execute successfully.

In [ ]:
!pip install groq -q
import os
from groq import Groq

# Re-initialize Groq client and model
os.environ["GROQ_API_KEY"] = "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" # Ensure your API key is correctly set
client = Groq(api_key = os.environ["GROQ_API_KEY"])
Model = "llama-3.1-8b-instant"
print("Groq client and model re-initialized.")

Groq client and model re-initialized.


In [ ]:
import io
import pandas as pd

# Re-create DataFrame 'df'
csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""
df=pd.read_csv(io.StringIO(csv_data))
print("DataFrame 'df' re-created.")

DataFrame 'df' re-created.


In [ ]:
import sqlite3
import pandas as pd # Need pandas for to_sql

# Re-establish database connection 'conn' and re-create 'students' table
conn = sqlite3.connect("college.db")
df.to_sql("students", conn, if_exists = "replace", index = False)
print("Database connection 'conn' re-established and 'students' table re-created.")

Database connection 'conn' re-established and 'students' table re-created.


In [ ]:
def text_to_sql_agent(user_question ,conn, client, model, verbose = True):
  print("="*60)

  print(f"User Question: {user_question}")
  print("="*60)

  if verbose:
    print("\n[Step 1] Reading database schema...")
  schema_text = get_schema(conn)
  if verbose:
    print("Schema loaded successfully")

  if verbose:
    print("\n[Step 2] Generating SQL query with Groq llm...")
  generated_sql = generate_sql(user_question, schema_text, client, model)
  if verbose:
    print(f"Generated SQL:\n {generated_sql}")
  if verbose:
    print("\n[Step 3] Executing SQL On the database...")
  result_df, error = execute_sql(generated_sql, conn)
  if error:
    print(f"Error executing SQL: {error}")
    return None, error

  if verbose:
    print(f"\n[Step 4] Query returned {len(result_df)} row(s)")
    print("\nResults")
    print("-"*40)
    print(result_df.to_string(index=False))
  print("-"*60)
  return result_df, generated_sql
result, sql_used = text_to_sql_agent(
    "Show top 5 students in Programming",
    conn, client, Model
)

User Question: Show top 5 students in Programming

[Step 1] Reading database schema...


NameError: name 'get_schema' is not defined

In [ ]:
!pip uninstall -y litellm crewai
!pip install -U "crewai[litellm]"

Found existing installation: litellm 1.83.0
Uninstalling litellm-1.83.0:
  Successfully uninstalled litellm-1.83.0
Found existing installation: crewai 1.14.3
Uninstalling crewai-1.14.3:
  Successfully uninstalled crewai-1.14.3
  Using cached crewai-1.14.6-py3-none-any.whl.metadata (36 kB)
  Using cached litellm-1.83.14-py3-none-any.whl.metadata (33 kB)
INFO: pip is looking at multiple versions of litellm to determine which version is compatible with other requirements. This could take a while.
  Using cached litellm-1.83.13-py3-none-any.whl.metadata (33 kB)
  Using cached litellm-1.83.12-py3-none-any.whl.metadata (33 kB)
  Using cached litellm-1.83.11-py3-none-any.whl.metadata (33 kB)
  Using cached litellm-1.83.10-py3-none-any.whl.metadata (33 kB)
  Using cached litellm-1.83.9-py3-none-any.whl.metadata (33 kB)
  Using cached litellm-1.83.8-py3-none-any.whl.metadata (33 kB)
  Using cached litellm-1.83.7-py3-none-any.whl.metadata (31 kB)
  Using cached aiohttp-3.13.5-cp312-cp312-manylin

In [ ]:
import os
from crewai import Agent, Task, Crew

os.environ["GROQ_API_KEY"] = "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" # Make sure to replace this with your actual Groq API key

def research_agent_example():
    researcher = Agent(
        role="Research Analyst",
        goal="Identify top AI trends for 2026",
        backstory="Experienced in AI research and development with 10+ years of industry knowledge",
        verbose=True,
        llm="groq/llama-3.1-8b-instant" # Pass model string directly, CrewAI will handle LiteLLM integration
    )

    task = Task(
        description="List and analyze the top 3 AI trends for 2026 with examples.",
        expected_output="A detailed explanation of each trend with examples.",
        agent=researcher
    )

    crew = Crew(
        agents=[researcher],
        tasks=[task],
        verbose=True
    )

    return crew.kickoff()

result = research_agent_example()
print(result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.14.6                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7fd8da07-6c1a-4faf-8932-12679044fdda                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: List and analyze the top 3 AI trends for 2026 with examples.                                             │
│  ID: 48fcea9b-8cdc-4d10-8b50-ad92c00919fb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│  Task: List and analyze the top 3 AI trends for 2026 with examples.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Top 3 AI Trends for 2026: Enabling Business Success and Innovation**                                         │
│                                                                                                                 │
│  As we approach 2026, the field of Artificial Intelligence (AI) continues to evolve at an unprecedented pace,   │
│  transforming industries and revolutionizing the way businesses operate. After years of research and            │
│  development, several trends are emerging that will have a significant impact on the AI landscape. As a         │
│  Research Analyst, I have identified the top 3 AI trends for 2026, along with examples of how they will shape   │
│  the future of business and innovation.                                                                         │
│                                                                                                                 │
│  **1. Explainability and Transparency in AI (Explainability AI)**                                               │
│                                                                                                                 │
│  Explainability AI is a rapidly emerging trend that focuses on developing AI systems that provide transparent   │
│  and interpretable decision-making processes. This trend is driven by the increasing need for trust and         │
│  accountability in AI decision-making, particularly in high-stakes applications such as healthcare, finance,    │
│  and law enforcement.                                                                                           │
│                                                                                                                 │
│  **Why it matters:** Explainability AI is crucial for mitigating the risks associated with AI, such as bias,    │
│  fairness, and accountability. By providing insights into AI decision-making processes, businesses can ensure   │
│  that their AI systems are fair, transparent, and explainable, leading to better decision-making and increased  │
│  trust from stakeholders.                                                                                       │
│                                                                                                                 │
│  **Examples:**                                                                                                  │
│                                                                                                                 │
│  * **Google's Explainable AI (XAI) toolkit:** Google has developed a suite of tools and frameworks to support   │
│  explainability in AI, including the What-If Tool, which allows developers to analyze and understand AI         │
│  decision-making processes.                                                                                     │
│  * **AI-powered chatbots with transparency:** Companies like IBM and Microsoft are developing chatbots that     │
│  provide transparent and explainable decision-making processes, enabling users to understand the reasoning      │
│  behind AI-driven responses.                                                                                    │
│                                                                                                                 │
│  **2. Multi-Echelon AI (MEA) for Complex Decision-Makin

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: List and analyze the top 3 AI trends for 2026 with examples.                                             │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Top 3 AI Trends for 2026: Enabling Business Success and Innovation**

As we approach 2026, the field of Artificial Intelligence (AI) continues to evolve at an unprecedented pace, transforming industries and revolutionizing the way businesses operate. After years of research and development, several trends are emerging that will have a significant impact on the AI landscape. As a Research Analyst, I have identified the top 3 AI trends for 2026, along with examples of how they will shape the future of business and innovation.

**1. Explainability and Transparency in AI (Explainability AI)**

Explainability AI is a rapidly emerging trend that focuses on developing AI systems that provide transparent and interpretable decision-making processes. This trend is driven by the increasing need for trust and accountability in AI decision-making, particularly in high-stakes applications such as healthcare, finance, and law enforcement.

**Why it matters:** Explainability AI is crucial for mitiga